# STAGE 7 - U-Net LEAKAGE DIAGNOSTIC (NON-COMPLIANT)

**WARNING: LEAKAGE DIAGNOSTIC - NOT GATE EVIDENCE - INVALID FOR EVALUATION.**

This run intentionally violates project gates at user request for diagnostic comparison only:
- Augments the whole dataset before splitting (leakage).
- Augmented variants of the same real chip may appear in both train and test.
- Uses a non-locked random train/test split on the augmented pool.
- Metrics are expected to be inflated/unreliable and MUST NOT be used as Stage 7 evidence, final results, or report claims.

Gate-compliant runs remain in `reports/stage7_unet_colab/` and `reports/stage7_unet_aug45/` (when complete).

In [ ]:
import json, os, random
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import rotate as ndrotate, shift as ndshift
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
INPUT_DIR = Path(os.environ.get('YAMUNA_STAGE7_INPUT', '/content/yamuna-stage7-input'))
CHIPS_DIR = Path(os.environ.get('YAMUNA_STAGE7_CHIPS', str(INPUT_DIR / 'chips_fixed')))
MASKS_DIR = Path(os.environ.get('YAMUNA_STAGE7_MASKS', str(INPUT_DIR / 'annotation' / 'masks')))
CHIPS_JSON = Path(os.environ.get('YAMUNA_STAGE7_CHIPS_JSON', str(INPUT_DIR / 'annotation' / 'chips.json')))
OUTPUT_DIR = Path(os.environ.get('YAMUNA_STAGE7_OUTPUT', '/content/stage7_unet_leakage_diagnostic'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EPOCHS = int(os.environ.get('STAGE7_EPOCHS', '40'))
BATCH_SIZE = int(os.environ.get('STAGE7_BATCH', '4'))
LR = 1e-3
SEED = 42
THRESH = 0.5
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('WARNING: LEAKAGE DIAGNOSTIC - NOT GATE EVIDENCE - INVALID FOR EVALUATION')
print('device', device)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=14, base=16):
        super().__init__()
        self.enc1, self.pool1 = ConvBlock(in_ch, base), nn.MaxPool2d(2)
        self.enc2, self.pool2 = ConvBlock(base, base*2), nn.MaxPool2d(2)
        self.enc3, self.pool3 = ConvBlock(base*2, base*4), nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base*4, base*8)
        self.up3, self.dec3 = nn.ConvTranspose2d(base*8, base*4, 2, 2), ConvBlock(base*8, base*4)
        self.up2, self.dec2 = nn.ConvTranspose2d(base*4, base*2, 2, 2), ConvBlock(base*4, base*2)
        self.up1, self.dec1 = nn.ConvTranspose2d(base*2, base, 2, 2), ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool1(e1)); e3=self.enc3(self.pool2(e2)); b=self.bottleneck(self.pool3(e3))
        d3=self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2=self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1=self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

def bce_dice(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob*target).sum(dim=(2,3))
    dice = (2*inter+1)/(prob.sum(dim=(2,3))+target.sum(dim=(2,3))+1)
    return bce + (1-dice).mean()

def compute_metrics(y_true, y_prob, threshold=THRESH):
    pred = y_prob >= threshold; truth = y_true.astype(bool)
    tp=int((pred&truth).sum()); fp=int((pred&~truth).sum()); fn=int((~pred&truth).sum()); tn=int((~pred&~truth).sum())
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    iou = tp/(tp+fp+fn) if tp+fp+fn else 0.0
    dice = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,precision=prec,recall=rec,f1=f1,iou=iou,dice=dice)

In [ ]:
def load_chip(e):
    p = CHIPS_DIR / e['quarter'] / f"{e['chip_id']}.tif"
    with rasterio.open(p) as ds: arr = ds.read().astype(np.float32)
    arr[:10] /= 10000.0; arr[10:] = arr[10:] / 5000.0 - 1.0
    return arr

def load_mask(e):
    p = MASKS_DIR / f"{e['chip_id']}_mask.png"
    with Image.open(p) as im: m = np.asarray(im.convert('L'), dtype=np.float32)
    if m.shape != (256,256): m = np.asarray(Image.fromarray(m.astype(np.uint8)).resize((256,256)), dtype=np.float32)
    return (m > 127).astype(np.float32)

def augment(x, y):
    """NON-COMPLIANT: used on whole dataset before split for leakage diagnostic only."""
    out = [(x.copy(), y.copy())]
    for angle in (45, 90, 135, 180):
        xr = ndrotate(x, angle, axes=(1,2), reshape=False, order=1, mode='reflect').astype(np.float32)
        yr = ndrotate(y, angle, axes=(0,1), reshape=False, order=0, mode='reflect')
        yr = (yr > 0.5).astype(np.float32)
        out.append((xr, yr))
    for dx, dy in ((-16,0),(16,0),(0,-16),(0,16)):
        xr = ndshift(x, shift=(0,dy,dx), order=1, mode='reflect').astype(np.float32)
        yr = ndshift(y, shift=(dy,dx), order=0, mode='reflect')
        yr = (yr > 0.5).astype(np.float32)
        out.append((xr, yr))
    return out

entries = json.loads(CHIPS_JSON.read_text(encoding='utf-8'))
print('mask entries', len(entries))
cache = {e['chip_id']:(load_chip(e), load_mask(e)) for e in entries}

In [ ]:
# NON-COMPLIANT: augment whole dataset, then split
Xall,Yall,ids=[],[],[]
for e in entries:
    x,y=cache[e['chip_id']]
    for j,(xa,ya) in enumerate(augment(x,y)):
        Xall.append(xa); Yall.append(ya); ids.append(f"{e['chip_id']}__aug{j}")
Xall=np.stack(Xall); Yall=np.stack(Yall)[:,None]; ids=np.asarray(ids)
idx=np.arange(len(Xall))
tr,te = train_test_split(idx, test_size=0.2, random_state=SEED, shuffle=True)
print('LEAKAGE DIAGNOSTIC total', Xall.shape, 'train', len(tr), 'test', len(te))
mean=Xall[tr].mean(axis=(0,2,3),keepdims=True); std=Xall[tr].std(axis=(0,2,3),keepdims=True)+1e-6
xtr=torch.from_numpy(((Xall[tr]-mean)/std).astype(np.float32)); ytr=torch.from_numpy(Yall[tr].astype(np.float32))
xte=torch.from_numpy(((Xall[te]-mean)/std).astype(np.float32)); yte_np=Yall[te][:,0]
loader=DataLoader(TensorDataset(xtr,ytr), batch_size=BATCH_SIZE, shuffle=True)
model=UNet(in_ch=14, base=16).to(device)
opt=torch.optim.Adam(model.parameters(), lr=LR)
for epoch in range(1,EPOCHS+1):
    model.train(); loss_sum=0.0
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad(); loss=bce_dice(model(xb),yb); loss.backward(); opt.step(); loss_sum+=loss.item()*len(xb)
    if epoch%10==0 or epoch==1:
        model.eval()
        with torch.no_grad(): p=torch.sigmoid(model(xte.to(device))).cpu().numpy()
        m=compute_metrics(yte_np,p)
        print(f"epoch {epoch:2d} loss {loss_sum/len(xtr):.4f} test_iou {m['iou']:.4f} test_dice {m['dice']:.4f}")
model.eval()
with torch.no_grad(): prob=torch.sigmoid(model(xte.to(device))).cpu().numpy()
m=compute_metrics(yte_np,prob)
metrics={
  'diagnostic':'LEAKAGE_DIAGNOSTIC_NOT_GATE_EVIDENCE_INVALID_FOR_EVALUATION',
  'stage':7,'model':'UNet','epochs':EPOCHS,'seed':SEED,'device':str(device),
  'timestamp_utc':datetime.now(timezone.utc).isoformat(),
  'n_real_entries':len(entries),'n_total_augmented':int(len(Xall)),'n_train':int(len(tr)),'n_test':int(len(te)),
  'split':'random 80/20 on augmented pool AFTER whole-dataset augmentation (LEAKAGE)',
  'augmentation':'real + rot45/90/135/180 + shifts ±16px = 9x, applied to WHOLE dataset before split (NON-COMPLIANT)',
  'test_metrics_leakage_inflated':m,
  'warning':'LEAKAGE DIAGNOSTIC - NOT GATE EVIDENCE - INVALID FOR EVALUATION. Metrics inflated due to pre-split augmentation/test contamination.'
}
(OUTPUT_DIR/'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame([m]).to_csv(OUTPUT_DIR/'test_metrics.csv', index=False)
print(json.dumps(m, indent=2))
print('LEAKAGE_DIAGNOSTIC_DONE')